In [73]:
import os
import numpy as np
import time

enrichment score ideas <br>
1. subramanian enrichment score with reduced data set
2. han enrichment score
3. p value from ks test
4. euclidean distance

In [34]:
path_parent = os.path.dirname(os.path.dirname(os.path.abspath('pairwise_gsea_association_mtx.ipynb')))
rank_mtx_f = open(path_parent + ('/deseq/res/rank_matrix.txt'),'r')
rank_mtx_raw = rank_mtx_f.readlines()
rank_mtx_f.close()

In [67]:
virus_names = rank_mtx_raw[0].strip().split('\t')
rank_dict = {}
for name in virus_names:
    rank_dict[name] = []

for line in rank_mtx_raw[1:]:
    line = line.strip().split('\t')
    for idx,name in enumerate(virus_names):
        rank_dict[name].append([line[0],int(line[idx+1])])

https://www.pathwaycommons.org/guide/primers/data_analysis/gsea/
![notation](sub_notation.png)
![sub_eq1](sub_eq1.png)
![sub_eq2_eq3](sub_eq2_eq3.png)
https://mathworld.wolfram.com/Supremum.html
![supremum](supremum.png)

In [69]:
#calculate enrichment score of signature gene set G of virus A in ranked list of genes L of virus B
#param rank_list: List([string,int]); list of tuples [gene, rank] ordered by rank 
#param G: List(string); list of signature genes of virus A
#param alpha: double; proportionality constant controls weight of rank
def subr_ES(rank_list,G,alpha=1):
    n = len(rank_list) #number of total genes
    nk = len(G) #number of genes in signature set
    L = [tup[0] for tup in rank_list] #ordered list of gene names
    s = [tup[1] for tup in rank_list] #ordered list of gene ranks
    
    #note: range(x,y) = [x,y) and lists are zero-indexed
    
    def Fi_Gk(i):
        numerator = sum([abs(s[t])**alpha * int(L[t] in G) for t in range(i)])
        denominator = sum([abs(s[t])**alpha * int(L[t] in G) for t in range(n)])
        return numerator/denominator
        
    def Fi_not_Gk(i):
        numerator = sum([1*int(L[t] not in G) for t in range(i)])
        denominator = n - nk
        return numerator/denominator
    
    return max([Fi_Gk(i) - Fi_not_Gk(i) for i in range(n)])

https://www.nature.com/articles/srep15820
![eq4](eq4.png)

In [70]:
#calculate the association score between virus A and virus B
#param rank_list_A: List([string,int]); list of tuples [gene, rank] ordered by rank for virus A
#param rank_list_B: List([string,int]); list of tuples [gene, rank] ordered by rank for virus B
#param sig_size: int; signature set contains the top {sig_size} and bottom {sig_size} ranked genes, sig_size = nk/2
def association_score(rank_list_A, rank_list_B,sig_size=250):
    L_A = [tup[0] for tup in rank_list_A] #ordered list of gene names of virus A
    L_B = [tup[0] for tup in rank_list_B] #ordered list of gene names of virus B
    
    upA = L_A[:sig_size] #first {sig_size} ranked genes of virus A
    downA = L_B[-sig_size:] #last {sig_size} ranked genes of virus A
    upB = L_A[:sig_size] #first {sig_size} ranked genes of virus B
    downB = L_B[-sig_size:] #last {sig_size} ranked genes of virus B
    
    return (subr_ES(rank_list_A,upB) + subr_ES(rank_list_B,upA) - subr_ES(rank_list_A,downB) - subr_ES(rank_list_B,downB))/4
    

In [75]:
def association_mtx(rank_dict):
    virus_names = list(rank_dict.keys())
    A = np.zeros(shape=(len(rank_dict),len(rank_dict))) #initialize the matrix
    
    t0 = time.time()
    
    for i in range(len(rank_dict)):
        for j in range(i+1,len(rank_dict)): #since the matrix is symmetric, don't calculate both A(i,j) and A(j,i)
            assoc_score = association_score(rank_dict[virus_names[i]], rank_dict[virus_names[j]])
            A[i,j] = assoc_score
            A[j,i] = assoc_score
        print(rank_dict[virus_names[i]] + f' {time.time()-t0}')
    return A

In [76]:
A = association_mtx(rank_dict)

KeyboardInterrupt: 